# Code file to run for SWRC simulations using MuMax3

In [ ]:
import numpy as np
import pandas as pd
import os
from glob import glob
from subprocess import run, PIPE, STDOUT

### Input function to be called. This function contains the MuMax3 script

In [ ]:
def ReservoirInput(Times,  Msat=1e6, Ku1=6.3e5, B_ext=[7.5e-3,0,0], Amplitude=1e6, pulsewidth=1.5e-9, frequency=0.75e9, transducer_radius=50e-9, FixDt=2e-13, Tmax=8e-9, save_m=True, save_table=False, save_snapshot=False):
    num_inputs = len(Times)
    script =  f"""
setgridsize(512, 512, 1 )
setcellsize(2e-9, 2e-9 ,1.5e-9)

disk := circle(1000e-9)
setgeom(disk)
defregion(0, disk)

Msat = {Msat}
Aex = 1.5e-11           // Exchange stiffness (J/m)
alpha = 0.012  
DisableZhangLiTorque = true
anisU = vector(0,0,1) 				// perpendicular anisotropy
Ku1 = {Ku1}

num_inputs := {num_inputs}
num_transducers := (num_inputs+1)*2
base_actuator := circle({transducer_radius}).transl(300e-9, 0, 0)
"""
    script += """
for i := 0; i < num_transducers; i++ { defregion(i+1, base_actuator.rotz(2*Pi*i/num_transducers)) }     

///////////////////////////////////
//      Initialize Magnetization //
/////////////////////////////////////

m = uniform(0, 0.01, 1)
for i := 0; i < num_transducers+1; i++ { Ku1.setRegion(i,  6.3e5)}
"""
    script += f"""
B_ext  = vector({B_ext[0]}, {B_ext[1]}, {B_ext[2]})
relax()
saveas(m,"M_Relax_Initial")

///////////////////////////////////
//       Running Parameters        //
/////////////////////////////////////

FixDt = {FixDt}              // Time step (seconds)
Tmax := {Tmax}             // Maximum time (seconds)

Amplitude := {Amplitude}
frequency := {frequency}
pulsewidth := {pulsewidth}

"""
    if save_snapshot: script += "autosnapshot(m, 100*FixDt)\n"
    if save_table: script += "tableautosave(100*FixDt)\n"
    if save_m: script += "autosave(m.Comp(2), 100*FixDt)\n"

    for i in range(num_inputs):
        script += f"Ku1.setRegion({i+1},  Amplitude * exp( - ((t-{Times[i]}-0.2e-9)/pulsewidth)*((t-{Times[i]}-0.2e-9)/pulsewidth) ) * sin(2*pi*frequency*(t-{Times[i]}-0.2e-9)) )\n"

    script += f"run(Tmax)\n"

    return script


In [ ]:
def ReadTable(filename):

    table = pd.read_table(filename)
    table.columns = " ".join(table.columns).split()[1::2]

    return table

def ReadFiles(outputdir):
    """Load all ovffiles in outputdir into a dictionary of numpy arrays
    with the ovffilename (without extension) as key"""

    p = run(
        ["mumax3-convert", "-numpy", outputdir + "/*.ovf"], stdout=PIPE, stderr=STDOUT
    )
    if p.returncode != 0:
        print(p.stdout.decode("UTF-8"))

    fields = {}
    for npyfile in glob(outputdir + "/*.npy"):
        key = os.path.splitext(os.path.basename(npyfile))[0]
        if not key.startswith("m"):
            continue
        fields[key] = np.load(npyfile)

    return fields

def Run(script, name, directory, verbose=False):

    os.makedirs(directory, exist_ok=True)  
    scriptfile = os.path.join(directory, name + ".txt")
    outputdir = os.path.join(directory, name + ".out")

    with open(scriptfile, "w") as f:
        f.write(script)

    p = run(["mumax3", "-f", scriptfile], stdout=PIPE, stderr=STDOUT)
    if verbose or p.returncode != 0:
        print(p.stdout.decode("UTF-8"))
    if os.path.exists(os.path.join(outputdir, "table.txt")):
        table = ReadTable(os.path.join(outputdir, "table.txt"))
    else:
        table = None

    fields = ReadFiles(outputdir)

    return table, fields

### The Compress class is used to convert the data intensive MuMax3 output files into csv files

In [ ]:
class Compress:
    def __init__(self, folder, directory):
        self.filepath = directory + "/" + folder
    def Condense(self, matrix, M):
        N = matrix.shape[0]
        if N % M != 0:
            print(f"Warning: {N} is not divisible by {M}. Cropping to {M * (N // M)}.")
            N = M * (N // M)  # Crop to largest multiple of M
            matrix = matrix[:N, :N]
        block_size = N // M 
        condensed = matrix.reshape(M, block_size, M, block_size).mean(axis=(1, 3))
        return condensed
    def Delete(self, filetype):
        if not os.path.exists(self.filepath):
            print(f"Error: The folder '{self.filepath}' does not exist.")
            return
        files = glob(os.path.join(self.filepath, f"*{filetype}"))
        if not files:
            print("No files found.")
            return
        for file in files:
            try:
                os.remove(file)
            except Exception as e:
                print(f"Failed to delete {file}: {e}")
    def Load(self):
        dataset = {}
        for filename in sorted(os.listdir(self.filepath)):  
            if filename.startswith("m_z000") and (filename.endswith(".npy") ):
                file_path = os.path.join(self.filepath, filename)
                try:
                    if filename.endswith(".npy"):
                        dataset[filename] = np.load(file_path)
                    elif filename.endswith(".ovf"):
                        dataset[filename] = np.loadtxt(file_path)  
                except Exception as e:
                    print(f"Error loading {filename}: {e}")
        return dataset
    def SaveCSV(self, matrix, filename):
        csv_path = os.path.join(self.filepath, filename)
        pd.DataFrame(matrix).to_csv(csv_path, index=False, header=False)
    def Proceed(self):
        matrices = self.Load()
        for filename, matrix in matrices.items():
            condensed_matrix = self.Condense(matrix[0][0][100:400, 100:400], 50)
            csv_filename = filename.replace(".npy", ".csv")
            self.SaveCSV(condensed_matrix, csv_filename)
        self.Delete('.ovf')
        self.Delete('.npy')

In [ ]:
def DataRun(input_data, filename):
    for j in range(5):
        for i in range(len(input_data)):
            Tinput = [ period*1e-10 for period in input_data[i]]
            filename += f"_X{i}"
            try:
                table1, fields = Run(ReservoirInput( Tinput, Msat=1e6, Amplitude=8e5, pulsewidth=1.25e-09, frequency=8e8, B_ext=[45e-3,0,0], Tmax=4e-9, save_table=True), filename)  
                Compress(f"{filename}.out").Proceed()
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                break
    print("Data run complete.")
